In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
%matplotlib widget

# First measurement

In [ ]:
f_tCBP = 1366 # mm
f_lens = 635 # mm
gamma = - f_lens / f_tCBP # Grandissement
print(gamma)

In [ ]:
wavelengths = np.array([385, 455, 505, 565, 660, 730, 810, 850, 940, 1050, 970, 780, 730, 625, 470])  # in nm

# Pin hole diameter = 2 mm
Input = np.array([16.7, 38.1, 12.4, 18.8, 40.7, 39.1, 59.9, 109.8, 49.3, 38.8, 52.2, 80.3, 48.9, 2.4, 2.1]) # Photodiode in the integrating sphere [µA]
Output = np.array([39, 89, 36.4, 57.3, 123.4, 112.3, 154.7, 275, 121, 104.3, 127.2, 214, 139.2, 7.86, 6.05])* 1e-3 # Photodiode after the lens [µA]

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(wavelengths, Input, 'o', color='blue')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Photodiode Current INPUT (µA)')
plt.legend()
plt.grid()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(wavelengths, Output, 'o', label='Output (After Lens)', color='orange')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Photodiode Current OUTPUT (µA)')
plt.legend()
plt.grid()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(wavelengths, Output/Input, 'o', ms=8, color='r')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Ratio Output/Input (Transmission)')
plt.legend()
plt.grid()

# Mesures automatisées

First acquisition.

In [ ]:
data_dir = '/home/lmousset/Projets_Recherche/LSST/Mesure_transmission_tcbp/'

mesures = [4, 5, 6]
nmeas = len(mesures)
#wavelengths = np.array([385, 455, 470, 505, 565, 625, 660, 730, 780, 810, 850, 940, 970, 1050])  # in nm
wls_led = np.array([385, 455, 505, 565, 660, 730, 780, 810, 850, 940, 970, 1050])  # in nm
nwls = len(wls_led)

In [ ]:
def get_data(data_dir, meas, wls):
    """Récupère les données de la mesure de transmission du tCBP pour une mesure donnée et une liste de longueurs d'onde."""
    data_path = data_dir + f'Mesure{meas}/'
    all_lens = []
    all_sphere = []
    for w, wl in enumerate(wls):
        print(f'Wavelength: {wl} nm')
        sphere = np.loadtxt(data_path + f'{wl:04d}/Sphere.txt')
        lens = np.loadtxt(data_path + f'{wl:04d}/Lens.txt')

        all_sphere.append(sphere)
        all_lens.append(lens)

    return all_sphere, all_lens

In [ ]:
# Compute ratios for all measurements
all_sphere_meas = {}
all_lens_meas = {}
all_ratios_meas = {}

for m in mesures:
    all_sphere_meas[m], all_lens_meas[m] = get_data(data_dir, meas=m, wls=wls_led)
    all_ratios = []
    print(f'\nMeasurement {m}')
    for w in range(nwls):
        size = len(all_sphere_meas[m][w])
        ratio = all_lens_meas[m][w][:size] / all_sphere_meas[m][w]
        all_ratios.append(ratio)
    all_ratios_meas[m] = all_ratios

In [ ]:
# Plot the data
m = 4
wl = 940
iwl = np.where(wls_led == wl)[0][0]

fig, axs = plt.subplots(1, 3, figsize=(12, 6))
axs = axs.flatten()
fig.suptitle(f'Measurement {m}, Wavelength {wl} nm')

axs[0].plot(all_sphere_meas[m][iwl]*1e6, '.')
axs[0].set_xlabel('Index')
axs[0].set_ylabel('Photodiode Current Input (µA)')


axs[1].plot(all_lens_meas[m][iwl]*1e9, '.')
axs[1].set_xlabel('Index')
axs[1].set_ylabel('Photodiode Current Output (nA)')

axs[2].plot(all_ratios_meas[m][iwl], '.')
axs[2].set_xlabel('Index')
axs[2].set_ylabel('Ratio (Lens/Sphere)')

fig.tight_layout()


In [ ]:
# Plot the ratios
plt.figure(figsize=(10, 6))
for m in mesures:
    plt.errorbar(wls_led, [np.mean(ratio) for ratio in all_ratios_meas[m]], yerr=[np.std(ratio) for ratio in all_ratios_meas[m]], fmt='o-', label=f'Mesure {m}')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Mean Ratio (I_out / I_in)')
plt.legend()
plt.grid()
plt.show()

# Divide by lens transmission and photodiode quantum efficiency

In [ ]:
# Charge les interpolateurs pour la photodiode NIST et la transmission de la lentille
with open(data_dir + 'Interp_pd_NIST_e_per_photon.pkl', 'rb') as file:
    interpolator_pd_NIST_e_per_photon = pickle.load(file)

with open(data_dir + 'Interp_lens_transmission.pkl', 'rb') as file:
    interpolator_lens = pickle.load(file)

with open(data_dir + 'Interp_pd_8D057_e_per_photon.pkl', 'rb') as file:
    interpolator_pd_8D057_e_per_photon = pickle.load(file)

transmission_lens_interp = interpolator_lens(wls_led)
pd_NIST_interp_e_per_photon = interpolator_pd_NIST_e_per_photon(wls_led)
pd_8D057_e_per_photon = interpolator_pd_8D057_e_per_photon(wls_led)

correction_factor = pd_8D057_e_per_photon / (pd_NIST_interp_e_per_photon * transmission_lens_interp)
#correction_factor = 1 / (pd_NIST_interp_e_per_photon * transmission_lens_interp)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(wls_led, transmission_lens_interp, 'o-', label='Lens Transmission')
plt.plot(wls_led, pd_NIST_interp_e_per_photon, 'o-', label='NIST Photodiode e/photon')
plt.plot(wls_led, pd_8D057_e_per_photon, 'o-', label='8D057 Photodiode e/photon')
plt.plot(wls_led, correction_factor, 'o-', label='Correction Factor')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Value')
plt.legend()
plt.grid()


In [ ]:
ratios_meas_corrected = np.zeros((nmeas, nwls))
for i, meas in enumerate(mesures):
    for w in range(nwls):
        ratios_meas_corrected[i, w] = np.mean(all_ratios_meas[meas][w])
    ratios_meas_corrected[i, :] *= correction_factor

In [ ]:
plt.figure(figsize=(10, 6))
for i, meas in enumerate(mesures):
    plt.plot(wls_led, ratios_meas_corrected[i, :]*100, 'o-', label=f'Mesure {meas}')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Transmission TCBP (%)')
plt.legend()
plt.grid()